# 01 - K3s Setup

Install a lightweight local Kubernetes cluster, using K3s

## Setup

In [1]:
import os

## Install

Run in the terminal:

```bash
    curl -sfL https://get.k3s.io | INSTALL_K3S_EXEC="\
    --disable=traefik \
    --disable=servicelb \
    --disable-cloud-controller \
    --kubelet-arg=cgroup-driver=systemd" \
    sh -
```
- _--disable=traefik_ - avoids auto-installing ingress (to install my own: Istio, NGINX, etc.)
- _--disable=servicelb_ - disables K3s’s basic klipper-lb (not GPU-aware)
- _--disable-cloud-controller_ - you’re running on bare metal, not in GKE/AKS
- _--cluster-cidr=10.244.0.0/16_ - tells K3s to allocate pod IPs from this subnet range for Pod networking (Flannel expects default 10.244.0.0/16, and the cluster was using 10.42.0.0/24)


<!-- --cluster-cidr=10.244.0.0/16" \ -->

Make the K3s kubeconfig available to CLI tools (Helm, kubectl, K9s, etc.)
Run in terminal:
```bash
    mkdir -p ~/.kube && \
    sudo cp /etc/rancher/k3s/k3s.yaml ~/.kube/config && \
    sudo chown $(id -u):$(id -g) ~/.kube/config
```

### Install Flannel (if it's missing)

- default Container Network Interface (CNI) plugin used by K3s
- provides pod-to-pod networking across nodes by assigning each pod a unique virtual IP and routing traffic between them

Run in terminal:
```bash
    kubectl apply -f https://raw.githubusercontent.com/flannel-io/flannel/master/Documentation/kube-flannel.yml
```

### CNI stuff

### Verify

In [8]:
!kubectl get nodes

NAME       STATUS   ROLES                  AGE   VERSION
laniakea   Ready    control-plane,master   28s   v1.33.3+k3s1


If the node is "Ready" and system pods are running - K3s cluster is healthy

## Test

Create a simple BusyBox pod that sleeps for 1 hour, to confirm the cluster can schedule workloads

In [3]:
%%writefile ../deployments/test-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: k3s-test-pod
spec:
  containers:
  - name: busybox
    image: busybox
    command: ["sleep", "3600"]

Overwriting ../deployments/test-pod.yaml


In [4]:
!kubectl apply -f ../deployments/test-pod.yaml

pod/k3s-test-pod created


In [5]:
!kubectl wait --for=condition=Ready pod/k3s-test-pod

pod/k3s-test-pod condition met


In [6]:
!kubectl get pod k3s-test-pod

NAME           READY   STATUS    RESTARTS   AGE
k3s-test-pod   1/1     Running   0          7s


### K9s (Optional)

If using K9s, link the K3s config so it works out of the box

```bash
    ln -s /etc/rancher/k3s/k3s.yaml ~/.kube/config
```


![K9s pods](../outputs/screenshots/01_k9s-pods.png)

![K9s pulses](../outputs/screenshots/01_k9s-pulses.png)

![K9s xray](../outputs/screenshots/01_k9s-xray-pods.png)

## Summary
- K3s installed and tested

## Cleanup

In [7]:
!kubectl delete pod k3s-test-pod

pod "k3s-test-pod" deleted


## Notes

Uinstall K3s:
```bash
     sudo /usr/local/bin/k3s-killall.sh && \
     sudo /usr/local/bin/k3s-uninstall.sh && \
     sudo rm -rf /etc/rancher /var/lib/rancher /run/k3s ~/.kube/config opt/cni/bin
```

Errors
- permission denied:
     ```bash
          unset KUBECONFIG
     ```